# Elicitation - GPT2 Model Family

gpt2, gpt2-medium, gpt2-large, gpt2-xl   

## Setup

In [3]:
# Cell 0: Environment Detection
import sys
from pathlib import Path
import torch

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Colab


In [5]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.9/239.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 133.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26

In [1]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

TransformerLens version: 2.18.0


In [2]:
# Cell 2: Project Root & Path Setup
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone -b rearrange-results {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Environment: Colab
Cloning into '/content/tmlr'...
remote: Enumerating objects: 1400, done.
remote: Counting objects: 100% (1400/1400), done.
remote: Compressing objects: 100% (858/858), done.
remote: Total 1400 (delta 644), reused 1272 (delta 518), pack-reused 0 (from 0)
Receiving objects: 100% (1400/1400), 10.22 MiB | 13.42 MiB/s, done.
Resolving deltas: 100% (644/644), done.
Project root: /content/tmlr


In [3]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import src
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Cell 5 - Model name variable
model_name = "gpt2-xl"

In [6]:
# Cell 6: Load Model
model = HookedTransformer.from_pretrained(f"{model_name}")

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-xl into HookedTransformer
Layers: 48
Heads: 25
Hidden size: 1600
Params: 1637.8M


In [7]:
print(torch.cuda.is_available(), next(model.parameters()).device)

True cuda:0


### Elicitation Battery

In [8]:
# Elicitation Battery — Loads prompts from a YAML file, runs them through
# a model, and saves results to a per-model CSV.
import importlib
import yaml
import pandas as pd

prompt_files = [
    #'accessibility.yaml',
    #'medical.yaml',
    #'legal.yaml',
    'finance.yaml',
    'control.yaml'
    ]

all_results = []

for prompts_file in prompt_files:
    domain = Path(prompts_file).stem
    prompts_path = PROJECT_ROOT / 'data' / prompts_file
    with open(prompts_path, 'r') as f:
        templates = yaml.safe_load(f)
    prompts = templates['prompts']
    print(f"\n--- Running {domain}: {len(prompts)} prompts ---")

    results = []
    for i, case in enumerate(prompts):
        print(f"\r  {i+1}/{len(prompts)}", end="")
        prompt = case['prompt']
        with torch.no_grad():
            full_output = model.generate(
                prompt,
                max_new_tokens=case['max_tokens'],
                temperature=0,
            )
        response = full_output[len(prompt):].strip()

        results.append({
            'domain': domain,          # <-- the missing key
            'prompt_id': case['prompt_id'],
            'concept': case['concept'],
            'prompt_type': case['prompt_type'],
            'template_type': case['template_type'],
            'prompt': prompt,
            'output': response,
            'max_tokens': case['max_tokens'],
            'model': model_name,
        })
    print()  # close the \r line so the next print doesn't collide

    domain_df = pd.DataFrame(results)
    out_path = PROJECT_ROOT / 'results' / f'{model_name}_{domain}.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    domain_df.to_csv(out_path, index=False)
    all_results.append(domain_df)

results_df = pd.concat(all_results, ignore_index=True)
print(f"\nSaved {len(results_df)} results to {output_path}")


--- Running finance: 44 prompts ---
  1/44

  0%|          | 0/100 [00:00<?, ?it/s]

  2/44

  0%|          | 0/100 [00:00<?, ?it/s]

  3/44

...

  43/44

  0%|          | 0/100 [00:00<?, ?it/s]

  44/44

  0%|          | 0/100 [00:00<?, ?it/s]

NameError: name 'output_path' is not defined

In [9]:
    domain_df = pd.DataFrame(results)
    out_path = PROJECT_ROOT / 'results' / f'{model_name}_{domain}.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    domain_df.to_csv(out_path, index=False)
    all_results.append(domain_df)

results_df = pd.concat(all_results, ignore_index=True)
print(f"\nSaved {len(results_df)} results to {out_path}")


Saved 132 results to /content/tmlr/results/gpt2-xl_control.csv


In [10]:
output_dir = PROJECT_ROOT / 'results' / 'elicitation' / 'gpt2' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / f'{model_name}-elicitation.md', 'w') as f:
  f.write(f"# Model data captured during Elicitation Battery\n\n")
  f.write(f"- Model name: {model_name}\n")
  f.write(f"- Model dtype: {next(model.parameters()).dtype}\n")
  f.write(f"- Layers: {model.cfg.n_layers}\n")
  f.write(f"- Heads: {model.cfg.n_heads}\n")
  f.write(f"- Hidden size: {model.cfg.d_model}\n")
  f.write(f"- Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M\n")



print(f"Saved to {output_dir}")

Saved to /content/tmlr/results/elicitation/gpt2/gpt2-xl


In [12]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/
!git commit -m "elicitation results: {model_name}"
!git push

On branch rearrange-results
Your branch is up to date with 'origin/rearrange-results'.

nothing to commit, working tree clean
Everything up-to-date


### Delete Model & Clear Cache

In [13]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

Memory cleared — GPU: 0.0GB allocated
